In [1]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.tabddpm.models import Tabddpm
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "magic.csv"
processed_file = ROOT / "preprocessed_data" / "magic_tabddpm.csv"
artifact_dir = ROOT / "artifacts"

print("ROOT:", ROOT)
print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=Tabddpm()
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="magic_tabddpm",
    artifact_store=store,
    model_name="tabddpm",
)

print(results)

ROOT: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic
Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\magic.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\magic.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\magic_tabddpm.csv
Loaded data with shape: (19020, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved dataset artifact under datasets/magic_tabddpm/split-20260808-120836
Step 100/200 | MLoss: 0.0000 | GLoss: 1.2416
Step 200/200 | MLoss: 0.0000 | GLoss: 0.1877
{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='magic_tabddpm', dataset_v

In [2]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "magic_tabddpm"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = []

continuous_cols = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\magic_tabddpm\split-20260808-120836
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
    fLength    fWidth     fSize     fConc    fConc1     fAsym   fM3Long  \
0  0.576572 -0.077991 -0.265128  1.028832  0.741567  0.954082  0.432571   
1  1.198148  1.499626  0.226059  0.454349  0.196015  0.372994  0.029714   
2  0.937586  0.661372  0.045905  0.692480  0.406868  0.506004  0.403314   
3  0.997582  0.861646  0.176028  1.012080  0.788913  0.851933  0.772559   
4  1.243345  0.260395 -0.153337  1.330008  1.085802  0.944656  0.731498   

   fM3Trans    fAlpha     fDist class  
0  0.999717  0.633620 -0.123610     g  
1  0.547992 -0.271923  0.070140     g  
2  0.913808 -0.552843  0.189329     g  
3  1.902783  0.163591  0.091474     g  
4  0.718656  0.988061 -0.042771     g  

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.